# Notebook 12 — Proyecto 1: DocBot

**Semana:** 12  
**Tipo:** Proyecto de portfolio  
**Tiempo estimado:** 5-6 horas

Este notebook no introduce conceptos nuevos: los integra todos.
Lo usas para verificar que DocBot funciona end-to-end, construir el testset
del proyecto y generar la tarjeta de métricas que mostrarás a clientes.

## Estructura del proyecto

```
proyectos/docbot/
├── app.py              ← interfaz Streamlit (ya creada)
├── docbot/
│   ├── chatbot.py      ← clase DocBot
│   ├── ingesta.py      ← pipeline de ingesta
│   ├── retrieval.py    ← pipeline de consulta
│   ├── memoria.py      ← sesiones + memoria semántica
│   └── evaluacion.py   ← evaluador de faithfulness
├── requirements.txt
└── README.md
```

## Antes de empezar

```bash
cd proyectos/docbot
pip install -r requirements.txt
```

El archivo `.env` debe estar en `proyectos/docbot/` con tu `OPENAI_API_KEY`.

---
## Celda 0 — Navegación al directorio del proyecto

In [ ]:
import os
import sys
from pathlib import Path

# Navega al directorio del proyecto para que los imports funcionen
PROYECTO_DIR = Path("../proyectos/docbot").resolve()
os.chdir(PROYECTO_DIR)
sys.path.insert(0, str(PROYECTO_DIR))

print(f"Directorio de trabajo: {os.getcwd()}")
print(f"Archivos del proyecto:")
for f in sorted(PROYECTO_DIR.rglob("*.py")):
    print(f"  {f.relative_to(PROYECTO_DIR)}")

---
## Celda 1 — Smoke test: DocBot funciona end-to-end

Verifica que todos los módulos importan correctamente y que el pipeline
completo funciona con una pregunta simple.

In [ ]:
from docbot import DocBot, RespuestaDocBot

# ── Inicialización ────────────────────────────────────────────────────────────
bot = DocBot(
    user_id="test_semana12",
    nueva_sesion=True,       # empieza limpio para la prueba
    evaluar_online=False     # sin evaluación online para el smoke test
)
print(f"DocBot iniciado | Sesión: {bot.session_id}")
print(f"Chunks indexados: {bot.n_docs}")

# ── Ingesta del corpus predefinido ────────────────────────────────────────────
FUENTES_TEST = [
    {"tipo": "wikipedia", "valor": "FastAPI",               "titulo": "FastAPI",     "categoria": "framework"},
    {"tipo": "wikipedia", "valor": "Pydantic",              "titulo": "Pydantic",    "categoria": "libreria"},
    {"tipo": "wikipedia", "valor": "JSON Web Token",        "titulo": "JWT",         "categoria": "seguridad"},
    {"tipo": "wikipedia", "valor": "Docker (software)",     "titulo": "Docker",      "categoria": "infra"},
    {"tipo": "wikipedia", "valor": "PostgreSQL",            "titulo": "PostgreSQL",  "categoria": "base_datos"},
    {"tipo": "wikipedia", "valor": "Representational state transfer", "titulo": "REST API", "categoria": "concepto"},
]

print("\nIniciando ingesta...")
stats = bot.ingestar(FUENTES_TEST)
print(f"\nEstadísticas de ingesta: {stats}")
print(f"Total chunks en colección: {bot.n_docs}")

# ── Smoke test: una pregunta ──────────────────────────────────────────────────
print("\nSmoke test — una pregunta simple:")
r = bot.responder("What is FastAPI?")

print(f"  Respondida:  {r.respondida}")
print(f"  Fuentes:     {r.fuentes}")
print(f"  Chunks:      {r.n_chunks}")
print(f"  Latencia:    {r.latencia_ms:.0f}ms")
print(f"  Respuesta:   {r.respuesta[:200]}...")

assert r.respondida, "ERROR: debería haber respondido sobre FastAPI"
assert r.n_chunks > 0, "ERROR: debería haber recuperado chunks"
print("\n✓ Smoke test pasado.")

---
## Celda 2 — Test de comportamiento: los 5 casos que todo RAG debe pasar

Un cliente te va a preguntar exactamente estos casos durante la demo.
Verifica que DocBot los maneja correctamente.

In [ ]:
import json

CASOS_CRITICOS = [
    {
        "id":       "caso_1_responde_bien",
        "tipo":     "respuesta correcta",
        "pregunta": "What is Docker and why is it used for application deployment?",
        "espera":   "respondida=True, menciona contenedores"
    },
    {
        "id":       "caso_2_no_inventa",
        "tipo":     "no alucina",
        "pregunta": "What is the exact revenue of Docker Inc in 2023?",
        "espera":   "respondida=False o respuesta muy conservadora (no puede saber el revenue)"
    },
    {
        "id":       "caso_3_fuera_corpus",
        "tipo":     "fuera de corpus",
        "pregunta": "What is the weather like in Madrid today?",
        "espera":   "respondida=False"
    },
    {
        "id":       "caso_4_seguimiento",
        "tipo":     "pregunta de seguimiento",
        "pregunta": "And what advantages does it have over traditional virtual machines?",
        "espera":   "responde sobre Docker (recuerda el contexto previo)"
    },
    {
        "id":       "caso_5_multi_fuente",
        "tipo":     "multi-fuente",
        "pregunta": "How does FastAPI use Pydantic for data validation?",
        "espera":   "cita tanto FastAPI como Pydantic como fuentes"
    },
]

print("Ejecutando los 5 casos críticos...\n")
resultados_casos = []

for caso in CASOS_CRITICOS:
    r = bot.responder(caso["pregunta"])
    resultado = {
        "id":        caso["id"],
        "tipo":      caso["tipo"],
        "pregunta":  caso["pregunta"][:60] + "...",
        "respondida": r.respondida,
        "fuentes":   r.fuentes,
        "n_chunks":  r.n_chunks,
        "latencia_ms": r.latencia_ms,
        "respuesta_preview": r.respuesta[:120] + "..."
    }
    resultados_casos.append(resultado)

    icono = "✓" if (caso["id"] in ("caso_1_responde_bien", "caso_4_seguimiento", "caso_5_multi_fuente") and r.respondida) \
               or (caso["id"] in ("caso_2_no_inventa", "caso_3_fuera_corpus") and not r.respondida) \
               else "?"

    print(f"{icono} [{caso['tipo']}]")
    print(f"  P: {caso['pregunta'][:70]}")
    print(f"  R: respondida={r.respondida} | fuentes={r.fuentes} | {r.latencia_ms:.0f}ms")
    print(f"  Espera: {caso['espera']}")
    print()

# Guarda los resultados
with open("evaluaciones/casos_criticos.json", "w") as f:
    json.dump(resultados_casos, f, indent=2, ensure_ascii=False)
print("Resultados guardados en evaluaciones/casos_criticos.json")

**Análisis de los casos críticos** (edita esta celda):

- Caso 2 (no alucina): ¿el sistema respondió o rechazó correctamente?
- Caso 3 (fuera de corpus): ¿devolvió el mensaje de "sin información"?
- Caso 4 (seguimiento): ¿entendió que la pregunta era sobre Docker?
- Caso 5 (multi-fuente): ¿citó ambas fuentes?
- ¿Qué caso falló o fue dudoso?

---
## Celda 3 — Evaluación formal del sistema

Genera un testset con preguntas representativas y evalúa faithfulness,
tasa de respuesta y relevancia del contexto.

In [ ]:
import statistics
from docbot.evaluacion import evaluar_turno, resumen_evaluaciones, ResultadoEval
from docbot.retrieval  import recuperar, UMBRAL_RELEVANCIA
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
cliente_eval  = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
chroma_eval   = chromadb.PersistentClient(path="./chroma_db")
coleccion_eval = chroma_eval.get_collection("rag_docs")

# ── Testset de 10 preguntas representativas ───────────────────────────────────
TESTSET = [
    # Preguntas dentro del corpus
    "What are the main features of FastAPI?",
    "How does JWT authentication work?",
    "What is the difference between Docker containers and virtual machines?",
    "What is Pydantic used for in Python?",
    "What are the ACID properties of PostgreSQL?",
    "What does REST stand for and what are its constraints?",
    "How does FastAPI handle automatic documentation generation?",
    "What is the history of PostgreSQL?",
    # Preguntas fuera del corpus (deben ser rechazadas)
    "What is the current price of Bitcoin?",
    "Who won the Champions League in 2024?",
]

print(f"Evaluando {len(TESTSET)} preguntas...")
print("(~3-4 llamadas LLM por pregunta — tomará ~2 minutos)\n")

evaluaciones_testset: list[ResultadoEval] = []

for i, pregunta in enumerate(TESTSET, 1):
    # Genera la respuesta
    r = bot.responder(pregunta)

    # Evalúa la faithfulness
    eval_r = evaluar_turno(
        pregunta=pregunta,
        respuesta=r.respuesta,
        chunks_textos=[c.texto for c in recuperar(pregunta, coleccion_eval, cliente_eval, k=4) if c.similitud >= UMBRAL_RELEVANCIA],
        chunks_scores=[c.similitud for c in recuperar(pregunta, coleccion_eval, cliente_eval, k=4) if c.similitud >= UMBRAL_RELEVANCIA],
        cliente=cliente_eval
    )
    evaluaciones_testset.append(eval_r)

    faith_str = f"{eval_r.faithfulness:.2f}" if eval_r.faithfulness is not None else "N/A"
    icono     = "✓" if eval_r.respondida else "○"
    print(f"  [{i:2d}] {icono} faithfulness={faith_str:4s} | {pregunta[:55]}...")

# ── Resumen de métricas ───────────────────────────────────────────────────────
resumen = resumen_evaluaciones(evaluaciones_testset)

print(f"\n{'='*55}")
print("TARJETA DE MÉTRICAS — DocBot v1.0")
print(f"{'='*55}")
print(f"  Preguntas evaluadas:    {resumen['n_total']}")
print(f"  Tasa de respuesta:      {resumen['tasa_respuesta']:.0%}")
print(f"  Faithfulness media:     {resumen.get('faithfulness_media', 'N/A')}")
print(f"  Context relevance:      {resumen.get('context_rel_media', 'N/A')}")
print(f"  Respuestas con alucin.: {resumen.get('n_con_alucinaciones', 0)}")
print(f"  Distribución veredictos: {resumen.get('distribucion_veredictos', {})}")
print(f"{'='*55}")

# Criterio de aprobado
faith = resumen.get('faithfulness_media')
tasa  = resumen['tasa_respuesta']
aprobado = faith is not None and faith >= 0.80 and tasa >= 0.70
print(f"\n{'✓ PROYECTO APROBADO' if aprobado else '✗ PROYECTO NECESITA MEJORAS'}")
if not aprobado:
    if faith is not None and faith < 0.80:
        print(f"  → Faithfulness {faith:.2f} < 0.80: refuerza el grounding en el prompt")
    if tasa < 0.70:
        print(f"  → Tasa de respuesta {tasa:.0%} < 70%: baja el umbral de relevancia o aumenta k")

# Guarda la tarjeta de métricas
import json
with open("evaluaciones/tarjeta_metricas_v1.json", "w") as f:
    json.dump({"version": "1.0", "fecha": __import__('datetime').date.today().isoformat(), **resumen}, f, indent=2)
print("\nTarjeta guardada en evaluaciones/tarjeta_metricas_v1.json")

---
## Celda 4 — Test de memoria entre sesiones

Verifica que la memoria semántica funciona: cierra la sesión,
abre una nueva y comprueba que el bot recuerda al usuario.

In [ ]:
# ── Sesión 1: el usuario se presenta ─────────────────────────────────────────
print("SESIÓN 1: el usuario se presenta")
print("=" * 55)

bot_s1 = DocBot(user_id="raffaele_portfolio", nueva_sesion=True)

mensajes_s1 = [
    "Hi, I'm Raffaele. I work at DIH DATAlife, an innovation hub in Galicia, Spain.",
    "I'm specializing in AI Engineering for the agri-food and biotech sector.",
    "What is Docker and could it be useful for deploying AI applications?",
]

for msg in mensajes_s1:
    r = bot_s1.responder(msg)
    print(f"\nTú: {msg}")
    print(f"Bot: {r.respuesta[:180]}..." if len(r.respuesta) > 180 else f"Bot: {r.respuesta}")

print(f"\nCerrando sesión 1...")
bot_s1.cerrar_sesion()
print(f"Sesión 1 ({bot_s1.session_id}) cerrada.")

# ── Sesión 2: ¿recuerda al usuario? ──────────────────────────────────────────
print("\n\nSESIÓN 2: nueva sesión — ¿el bot recuerda?")
print("=" * 55)

bot_s2 = DocBot(user_id="raffaele_portfolio")  # carga la sesión más reciente

mensajes_s2 = [
    "Hey, I'm back! Do you remember anything about me?",
    "Given my domain (agri-food and biotech), would you recommend FastAPI or Django REST?",
]

for msg in mensajes_s2:
    r = bot_s2.responder(msg)
    print(f"\nTú: {msg}")
    print(f"Bot: {r.respuesta[:250]}..." if len(r.respuesta) > 250 else f"Bot: {r.respuesta}")

# ── Verifica ──────────────────────────────────────────────────────────────────
primera_respuesta = bot_s2.responder("Do you remember where I work?")
menciona_datalife = any(kw in primera_respuesta.respuesta.lower()
                        for kw in ["datalife", "galicia", "innovation", "hub", "spain"])

print(f"\n{'='*55}")
print(f"Test memoria: {'✓ PASADO' if menciona_datalife else '✗ FALLIDO'}")
print(f"  La respuesta {'menciona' if menciona_datalife else 'NO menciona'} datos de la sesión anterior")
bot_s2.cerrar_sesion()

---
## Celda 5 — Arrancar la interfaz Streamlit

Cuando el código pasa todos los tests anteriores, arranca la interfaz.
Úsala para hacer la demo al cliente.

In [ ]:
# Instrucciones para arrancar la app
print("Para arrancar DocBot como aplicación web:")
print()
print("  cd proyectos/docbot")
print("  streamlit run app.py")
print()
print("Abre http://localhost:8501 en tu navegador.")
print()
print("Checklist pre-demo:")
print("  [ ] La app arranca sin errores")
print("  [ ] La ingesta de 3+ documentos funciona")
print("  [ ] El chat responde preguntas sobre los documentos")
print("  [ ] Las preguntas fuera de corpus muestran el mensaje de 'no información'")
print("  [ ] El historial persiste al refrescar la página")
print("  [ ] La tarjeta de métricas muestra faithfulness ≥ 0.80")

---
## Reto: adapta el corpus a tu dominio

El corpus de Wikipedia es el fallback gratuito. Para que este proyecto
tenga valor real de portfolio, adáptalo a tu dominio (agri-food, biotech, forestal).

**Opción A — Wikipedia en español/inglés sobre tu dominio:**

In [ ]:
# Corpus de dominio: agri-food y biotech (relevante para DIH DATAlife)
# Descomenta y ejecuta para indexar un corpus más representativo

# CORPUS_DATALIFE = [
#     # Agri-food
#     {"tipo": "wikipedia", "valor": "Precision agriculture",         "titulo": "Precision Agriculture",   "categoria": "agri-food"},
#     {"tipo": "wikipedia", "valor": "Food traceability",             "titulo": "Food Traceability",        "categoria": "agri-food"},
#     {"tipo": "wikipedia", "valor": "Aquaculture",                   "titulo": "Aquaculture",              "categoria": "agri-food"},
#     {"tipo": "wikipedia", "valor": "Common Fisheries Policy",       "titulo": "Common Fisheries Policy",  "categoria": "agri-food"},
#     # Biotech
#     {"tipo": "wikipedia", "valor": "Bioinformatics",                "titulo": "Bioinformatics",           "categoria": "biotech"},
#     {"tipo": "wikipedia", "valor": "CRISPR",                        "titulo": "CRISPR",                   "categoria": "biotech"},
#     {"tipo": "wikipedia", "valor": "Synthetic biology",             "titulo": "Synthetic Biology",        "categoria": "biotech"},
#     # Forestal
#     {"tipo": "wikipedia", "valor": "Sustainable forest management", "titulo": "Sustainable Forestry",    "categoria": "forestal"},
#     {"tipo": "wikipedia", "valor": "Forest Stewardship Council",    "titulo": "FSC Certification",        "categoria": "forestal"},
# ]

# bot_dominio = DocBot(user_id="docbot_datalife", coleccion_docs="rag_datalife", nueva_sesion=True)
# stats = bot_dominio.ingestar(CORPUS_DATALIFE)
# print(stats)

**Opción B — Tus propios documentos en Markdown:**

```python
# Coloca tus documentos en proyectos/docbot/datos/
# Ejemplo: guías técnicas, informes, normativa

from pathlib import Path

mis_docs = [
    {"tipo": "markdown", "valor": str(p), "titulo": p.stem, "categoria": "interno"}
    for p in Path("datos").glob("*.md")
]
print(f"{len(mis_docs)} documentos encontrados en datos/")

if mis_docs:
    bot_propio = DocBot(user_id="mi_usuario", coleccion_docs="rag_propio", nueva_sesion=True)
    stats = bot_propio.ingestar(mis_docs)
    print(stats)
```

---
## Reflexión post-proyecto

Dedica 30 minutos a responder estas preguntas antes de pasar a la semana 13.

1. **¿Cuál fue la parte más difícil de integrar?**  
   ...

2. **¿Qué pregunta del testset falló con más frecuencia y por qué?**  
   ...

3. **Si un cliente del sector agri-food te pidiera este sistema, ¿qué le cobrarías y en cuánto tiempo lo entregarías?**  
   ...

4. **¿Qué funcionalidad añadirías primero con una semana extra?**  
   ...

5. **¿Qué parte del código reutilizarías directamente para el Proyecto 2?**  
   ...

---
## Cierre del bloque de Agentes (S9-S12)

Completaste las cuatro semanas del bloque:

- **S09** Agentes ReAct con herramientas
- **S10** Memoria y estado entre sesiones
- **S11** Planificación y sistemas multi-agente
- **S12** Proyecto 1: DocBot ← aquí

La semana 13 empieza el **Bloque 6: Despliegue y producción** —
cómo convertir DocBot en una API REST con FastAPI, dockerizarla
y desplegarla en la nube.

---
*Notebook 12 — Semana 12 del Plan de Formación AI Engineering*